# Dynamic Time Warping

In [ ]:
#| eval: false

from fhemb.utils.cutils import depict_DTW, calc_dm_accuracy, plot_dm_heatmap, plot_dm_projection, svd_dmatrix, depict_accuracy

/Users/radned/.pyenv/versions/p311.fhemb/lib/python3.11/site-packages/threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
2026-04-25 03:02:34.224933: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
DEBUG:fhemb.config.settings:Loading environment from /Users/radned/.config/fhemb/.env.paths
DEBUG:fhemb.config.s

In [ ]:
#| hide
from nbdev import show_doc

## DTW Alignment
> is based on distance metrics

### DTW alignment between two time series

In [ ]:
#| eval: false
show_doc(depict_DTW, name="Depict DTW of a pair of time series",  title_level=4)

---

#### Depict DTW of a pair of time series

```python

def depict_DTW(
    x1:ndarray,
    x2:ndarray, # Time series where the first dimension is time and the second dimension are the features.
    dist:Callable=euclidean, # Distance metric function to use for DTW.
    radius:int=1, # Radius parameter for the FastDTW algorithm.
):


```

*Use `fastdtw` to visualize the DTW alignment between two time series.*

::: {.callout collapse="true" title= "DTW Distance Metrics - Summary Table"}
| `dist` value        | Function                | Equation | Description |
|---------------|--------------------------|----------|-------------|
| **Euclidean** | `euclidean(u, v)`        | $\sqrt{\sum_i (u_i - v_i)^2}$ | Standard L2 distance. |
| **Manhattan** | `cityblock(u, v)`        | $\sum_i \|u_i - v_i\|$ | L1 distance; sum of absolute differences. |
| **Cosine**    | `cosine(u, v)`           | $1 - \frac{u \cdot v}{\|u\|\|v\|}$ | Distance based on cosine similarity. |
| **Mahalanobis** | `mahalanobis(u, v, VI)` | $\sqrt{(u-v)^\top V^{-1}(u-v)}$ | Distance scaled by covariance structure. |
| **Squared Euclidean** | `sqeuclidean(u, v)` | $\sum_i (u_i - v_i)^2$ | L2 distance without the square root. |
| **Chebyshev** | `chebyshev(u, v)`        | $\max_i \|u_i - v_i\|$ | Maximum coordinate difference. |
| **Minkowski** | `minkowski(u, v, p)`     | $\left(\sum_i \|u_i - v_i\|^p\right)^{1/p}$ | Generalized Lp distance. |
| **Bray–Curtis** | `braycurtis(u, v)`      | $\frac{\sum_i \|u_i - v_i\|}{\sum_i \|u_i + v_i\|}$ | Normalized L1 emphasizing relative differences. |
| **Canberra**  | `canberra(u, v)`         | $\sum_i \frac{\|u_i - v_i\|}{\|u_i\| + \|v_i\|}$ | Weighted L1 emphasizing small values. |
| **Correlation** | `correlation(u, v)`     | $1 - \text{corr}(u, v)$ | Distance based on linear correlation. |
| **Jaccard**   | `jaccard(u, v)`          |  $1 - \frac{\|u \cap v\|}{\|u \cup v\|}$ | For binary/boolean vectors. |
| **Hamming**   | `hamming(u, v)`          | $\frac{1}{n}\sum_i [u_i \ne v_i]$ | Fraction of differing coordinates. |
:::

::: {.callout-note collapse="true" title="FastDTW: radius value"}
The `radius` parameter controls how wide the refinement window is around the projected DTW path in FastDTW.  
It must be a **non‑negative integer** (`0, 1, 2, …`). Larger values improve accuracy at the cost of speed.

| `radius` value | Interpretation | Effect |
|--------------|----------------|--------|
| `0`          | Follow projected path exactly; no refinement | Fastest, least accurate |
| `1`          | Slight refinement around the path | Still fast; modest accuracy gain |
| `2`          | Balanced refinement window | Good trade‑off between speed and accuracy |
| `≥ 5`        | Wide refinement window | Much slower; very close to exact DTW |
:::


::: {.callout-caution title="radius value"}
Applies to `fastdtw` only.
:::

### DTW alignment between multiple time series

In [ ]:
#| eval: false

show_doc(plot_dm_heatmap, title_level=4, name="Depict distance matrix as a heatmap")

---

#### Depict distance matrix as a heatmap

```python

def plot_dm_heatmap(
    ts, # Array of shape (n_samples, n_features, n_timesteps).
    alignment_type:str='fastdtw', # Type of alignment method to use. Options:
'dcor' or DTW type ('fastdtw', 'tslearn_dtw', 'soft_dtw', 'soft_dtw_div').
    gamma:int=1, # Smoothing parameter for Soft-DTW (only used for DTW-based alignment types):
- Small gamma → behaves like classic DTW (hard minimum)
- Large gamma → smoother, more diffused alignment
    dist:function=euclidean, # Distance metric function used inside DTW. Only applies when alignment_type='fastdtw'.
Ignored for 'tslearn_dtw' (uses euclidean), 'soft_dtw'/'soft_dtw_div' (use squared euclidean),
and 'dcor'.
    p:int=3, # Parameter for Minkowski distance (if used). Only applies to DTW-based alignment types.
    radius:int=1, # Radius parameter for the FastDTW algorithm. Only applies when alignment_type='fastdtw'.
    normalize:NoneType=None, # Normalization method for the distance matrix.
Options: 'max', 'minmax', 'z-score', 'l1', 'l2', 'softmax', 'sym', 'log', 'none'.
    n_jobs:int=4, # Number of parallel jobs.
    labels:NoneType=None, # Labels for the time series. If None, defaults to TS0, TS1, ...
    title:str='Distance Matrix Heatmap', # Title of the plot.
    color_continuous_scale:str='Viridis', # Colormap for the heatmap.
    base_size:int=20, # Base pixel size per time series (controls figure dimensions).
): # Displays an interactive Plotly heatmap.


```

*Visualize a distance matrix as an interactive Plotly heatmap with hover tooltips.*
The distance matrix is computed using either distance correlation (dcor) or a DTW-based
method depending on the selected alignment type.

::: {.callout collapse="true" title= "DTW Methods - Equations Summary (tslearn + fastdtw)"}
| `alignment_type` value | Underlying Function / Class                | Conceptual Metric        | Equation / Definition | Notes |
|------------------|--------------------------------------------|---------------------------|------------------------|-------|
| `tslearn_dtw`    | `tslearn_dtw(ts1, ts2)`                    | Dynamic Time Warping (DTW) | $$\mathrm{DTW}(X,Y) = \min_{\pi} \sum_{(i,j)\in\pi} \|X_i - Y_j\|$$ | Exact DTW; dynamic programming; $O(n^2)$. |
| `fastdtw`        | `fastdtw(ts1, ts2)`                        | FastDTW (approx. DTW)      | *No closed form* (approximation of DTW) | Linear‑time, linear‑space approximation; very fast. |
| `soft_dtw`       | `tslearn_soft_dtw(ts1, ts2, gamma)`        | Soft‑DTW                   | $$\text{SoftDTW}_\gamma(X,Y) = -\gamma \log \sum_{\pi} e^{-\frac{1}{\gamma} \sum_{(i,j)\in\pi}\|X_i-Y_j\|}$$ | Differentiable relaxation of DTW; controlled by `gamma`. |
| `soft_dtw_div`   | `soft_dtw_divergence(ts1, ts2, gamma)`     | Soft‑DTW Divergence        | $$D(X,Y) = \mathrm{SoftDTW}(X,Y) - \tfrac{1}{2}(\mathrm{SoftDTW}(X,X) + \mathrm{SoftDTW}(Y,Y))$$ | Proper divergence; symmetric; $\geq 0$. |
| `dcorrelation` | `dcor.distance_correlation(x, y)` | Distance Correlation (biased V‑statistic) | $$\mathrm{dCor}(X,Y)=\frac{\mathrm{dCov}(X,Y)}{\sqrt{\mathrm{dCov}(X,X)\,\mathrm{dCov}(Y,Y)}}$$ with $$\mathrm{dCov}^2(X,Y)=\frac{1}{n^2}\sum_{i,j} A_{ij}B_{ij}$$ where \(A,B\) are doubly‑centered distance matrices. | Detects any dependence; 0 iff independent; supports multivariate inputs; exponent ∈ (0,2]; multiple computation methods; \(O(n^2)\). |


:::


::: {.callout-caution title="dist value"}
Applies to `fastdtw` only.
:::

::: {.callout collapse="true" title= "Matrix Normalization Methods — API Summary"}
| `normalize` value | Meaning / Transformation | Equation | Scope | Strengths | Notes |
|-------------|--------------------------|----------|--------|-----------|-------|
| `max` | Global max normalization | $M \leftarrow M / \max(\|M\|)$ | Whole matrix | Simple global scaling; preserves sign | Sensitive to outliers; collapses if max is 0 |
| `minmax` | Min–max scaling to $[0,1]$ | $M \leftarrow \dfrac{M - \min(M)}{\max(M) - \min(M)}$ | Whole matrix | Uniform scaling; preserves relative ordering | If all values equal → returns zeros |
| `z-score` | Standardization (zero mean, unit variance) | $M \leftarrow \dfrac{M - \mu}{\sigma}$ | Whole matrix | Centers + scales; good for Gaussian‑like data | If $\sigma = 0$ → returns zeros |
| `l1` | Row‑wise L1 normalization | $M_{i,:} \leftarrow \dfrac{M_{i,:}}{\sum_j \|M_{i,j}\|}$ | Row‑wise | Produces rows summing to 1; good for sparse data | Zero rows remain zero (safe handling) |
| `l2` | Row‑wise L2 normalization | $M_{i,:} \leftarrow \dfrac{M_{i,:}}{\sqrt{\sum_j M_{i,j}^2}}$ | Row‑wise | Normalizes direction; common in embeddings | Zero rows remain zero (safe handling) |
| `softmax` | Row‑wise softmax (probabilities) | $M_{i,j} \leftarrow \dfrac{e^{M_{i,j}}}{\sum_k e^{M_{i,k}}}$ | Row‑wise | Converts rows into probability distributions | Uses max‑shift for numerical stability |
| `sym` | Symmetric graph normalization | $M \leftarrow D^{-1/2} \, M \, D^{-1/2}$ where $D_{ii} = \sum_j M_{ij}$ | Whole matrix | Standard in spectral graph theory; stabilizes adjacency matrices | Adds $1e{-8}$ to avoid division by zero |
| `log` | Logarithmic transform | $M \leftarrow \log(1 + M)$ | Element‑wise | Compresses large values; reduces skew | Requires non‑negative input for meaningful interpretation |
| `None` | No normalization | — | — | Leaves matrix unchanged | Useful for debugging or raw comparisons |
:::

::: {.callout-note collapse="true" title="gamma value"}
The **gamma ($\gamma$)** parameter controls how quickly similarity decays with distance in the Gaussian/RBF kernel. It effectively sets the *locality scale* of the embedding: small $\gamma$ captures global structure, large $\gamma$ emphasizes local neighborhoods. Here, $\mathbf{d}$ denotes a characteristic distance scale, e.g. the **median** of all pairwise distances in the distance matrix.

| $\gamma$ value (relative to distance scale) | Kernel Width | Effect on Similarity Matrix | Effect on 3D Embedding | When to Use |
|--------------------------------------|--------------|------------------------------|-------------------------|--------------|
| **Very small** ($\gamma ≈ 0.001 / d^2$)      | Very wide    | Most distances map to high similarity (matrix almost uniform) | Embedding collapses; points cluster together | When distances are extremely noisy; rarely useful |
| **Small** ($\gamma \approx 0.01 / d^2$)            | Wide         | Slow decay; far points still moderately similar | Smooth, global structure; weak cluster separation | When you want global geometry preserved |
| **Medium** ($\gamma \approx 0.1 / d^2$)            | Balanced     | Reasonable decay; neighbors clearly more similar | Balanced embedding: clusters visible, global shape preserved | **Good default** when scale is unknown |
| **Large** ($\gamma\approx 1 / d^2$)               | Narrow       | Only close neighbors have high similarity | Strong cluster separation; global shape distorted | When clusters matter more than global layout |
| **Very large** ($\gamma\geq 10 / d^2$)         | Very narrow  | Almost all similarities ≈ 0 except identical points | Embedding becomes noisy; overfitting to local noise | Only for extremely tight local structure |
:::

::: {.callout-caution title="gamma value"}
Applies to `soft_dtw` only.
::: 

### DTW alignment geometry

In [ ]:
#| eval: false

show_doc(plot_dm_projection, title_level=4, name="Distance matrix projection using a dimensionality reduction method")

---

#### Distance matrix projection using a dimensionality reduction method

```python

def plot_dm_projection(
    ts:VAR_POSITIONAL, # Arrays of shape (n_samples, n_features, n_timesteps)
    alignment_type:str='fastdtw', # Type of alignment method to use: 'dcor' or DTW type ('fastdtw', 'tslearn_dtw', 'soft_dtw', 'soft_dtw_div').
    gamma:int=1, # smoothing parameter (for soft_dtw only):
    Small gamma → behaves more like classic DTW (hard minimum)
    Large gamma → smoother, more diffused alignment (less sensitive to exact path)
    0.01 Very close to DTW:     Precise alignment
    1.0 Balanced smoothing:     Most common default
    10.0 Very smooth: Robust to noise, good for optimization
    dist:function=euclidean, # Distance metric function to use for DTW is implicitly euclidean. Only applies for alignment_type='fastdtw'.
    p:int=3, # Parameter for Minkowski distance (if used).
    radius:int=1, # Radius parameter for the FastDTW algorithm.
    projection:str='mds', # Distance Matrix MDS Projection or PCA or TSNE
    transformation:str='gauss', # Method to transform distances into similarities for PCA.
    Options: 'gauss', 'invert', 'minmax'
    normalize:NoneType=None, # Normalization method to apply to the similarity matrix.
    Options: 'max', 'minmax', 'z-score', 'l1', 'l2', 'softmax', 'sym', 'none'
    n_jobs:int=4, # number of jobs for parallel execution
    labels:NoneType=None, # labels for each time series.
    title:str='Distance Matrix Projection', # plot title
    dimensions:int=2, # 2 or 3 for projection
):


```

*Visualize distance matrix using MDS, TSNE, or PCA (2D or 3D projection) with Plotly.*
The distance matrix is calculated using distance correlation (dcor) or a DTW method.

::: {.callout collapse="true" title= "Dimensionality Reduction - Summary Table"}
| `projection` value | Input Type | What It Optimizes | Strengths | Weaknesses | Typical Use |
|--------|------------|-------------------|-----------|------------|-------------|
| **MDS (Multidimensional Scaling)** | Distance matrix (precomputed) | Preserves pairwise distances | Stable, interpretable, works directly on distances | Can struggle with nonlinear structure | Visualizing global geometry of distance-based data |
| **t‑SNE** | Distance matrix (precomputed) | Preserves local neighborhoods | Excellent for clusters and nonlinear manifolds | Distorts global structure; parameter‑sensitive | Exploring clusters and local structure |
| **PCA** | Feature matrix (or similarity derived from distances) | Maximizes variance along orthogonal axes | Fast, deterministic, widely understood | Linear only; not naturally distance‑based | Baseline dimensionality reduction and comparison |
:::

::: {.callout collapse="true" title= "Similarity Transformations for PCA- Summary Table"}
| `transformation` value | Formula | Intuition | Strengths | Notes |
|--------|---------|-----------|-----------|--------|
| **Gaussian (RBF) Similarity** | $K_{ij} = \exp\!\left(-\frac{D_{ij}^2}{2\sigma^2}\right)$ | Nearby points get high similarity; far points decay smoothly | Handles nonlinear structure; tunable locality via $\sigma$ | Most common kernel for PCA on distance‑based data |
| **Inverse Distance Similarity** | $K_{ij} = \frac{1}{1 + D_{ij}}$ | Similarity decreases monotonically with distance | Simple, bounded, interpretable | Good when distances vary widely |
| **Min‑Max Normalized Similarity** | $K_{ij} = 1 - \frac{D_{ij} - D_{\min}}{D_{\max} - D_{\min}}$ | Linearly maps distances to $[0,1]$ | Preserves rank; easy to visualize | Useful when PCA expects normalized inputs |
:::

::: {.callout-caution title="transformation value"}
Applies to PCA only.
::: 

#### Intrinsic dimensionality of a distance matrix via singular value decomposition (SVD)
> Compute Singular Value Decomposition of a Distance Matrix

In [ ]:
#| eval: false

show_doc(svd_dmatrix, title_level=4)

---

#### svd_dmatrix

```python

def svd_dmatrix(
    ts, # Arrays of shape (n_samples, n_timesteps) or (n_samples, n_features, n_timesteps).
    alignment_type:str='fastdtw', # Type of alignment method to use ('dcor', 'fastdtw', 'tslearn_dtw', 'soft_dtw', 'soft_dtw_div').
    gamma:int=1, # smoothing parameter (for soft_dtw only):
    Small gamma → behaves more like classic DTW (hard minimum)
    Large gamma → smoother, more diffused alignment (less sensitive to exact path)
    0.01 Very close to DTW:     Precise alignment
    1.0 Balanced smoothing:     Most common default
    10.0 Very smooth: Robust to noise, good for optimization
    dist:function=euclidean, # Distance metric function to use for DTW (only for fastdtw).
    p:int=3, # Parameter for Minkowski distance (if used).
    radius:int=1, # Radius parameter for the FastDTW algorithm.
    transformation:str='gauss', # Method to transform distances into similarities.
    Options: 'gauss', 'invert', 'minmax'
    normalize:NoneType=None, # Normalization method to apply to the similarity matrix.
    Options: 'max', 'minmax', 'z-score', 'l1', 'l2', 'softmax', 'sym', 'none'
    n_jobs:int=4, # number of jobs for parallel execution
): # U: Left singular vectors
S: Singular values (1D array)
Vt: Transpose of right singular vectors


```

*Compute the Singular Value Decomposition (SVD) of a similarity matrix derived from time series distances.*
The distance matrix is calculated using distance correlation (dcor) or a DTW method.

## Classification Accuracy
> of a binary classification problem. A class contains landmark embeddings (rows in a DTW distance matrix) of time series that are synchronized, i.e., belong to the same segment the underlying `Datasource` object.

In [ ]:
#| eval: false

show_doc(calc_dm_accuracy, title_level=3, name="Calculate and store")

---

### Calculate and store

```python

def calc_dm_accuracy(
    Xopera0, Xopera1, dirname, # Directory in which the results Excel file will be stored.
    alignment_type:str='fastdtw', # alignmeng methods to use ('dcor', 'fastdtw', 'tslearn_dtw', 'soft_dtw', 'soft_dtw_div').
    normalize:bool=False, # Whether to normalize the data ('minmax' or 'meanvar').
    radius:int=1, # Radius parameter for fastdtw.
    feature:NoneType=None, # Feature index to include in the output filename.
    random_states:int=1, # Number of random train–test splits to evaluate.
)->Path: # Path to the Excel file containing the computed accuracy results.
The file is guaranteed to exist upon return.
Path = ROOT/dirname/acc_summ.xlsx or acc_summ_{feature}.xlsx if feature is specified.


```

*Compute classification accuracies across multiple classifiers and distance_matrix-based embeddings, and write the aggregated results to an Excel file.*

The output directory is created if missing. The Excel file is always written before the function returns and contains:
- a combined summary of all classifiers,
- metadata describing the evaluation setup,
- per-classifier accuracy summaries and split-level results.

::: {.callout collapse="true" title= "Classifier-Based Similarity Functions - Summary Table"}
| Classifier name | Classifier | Key Hyperparameters | Input Requirements | Output | Strengths | Notes |
|-----------------|------------|---------------------|--------------------|--------|-----------|-------|
| `svm` | `sklearn.svm.SVC` | `kernel ∈ {linear, poly, rbf, sigmoid}`, `C` | Feature matrices (`X_train`, `X_test`), label vectors | Accuracy ∈ [0, 1] | Strong for linear or smooth nonlinear boundaries | Standard SVM; no probability outputs |
| `knn` | `KNeighborsClassifier (k=3)` | `metric ∈ KNN_METRICS`, `p` for Minkowski | Raw features or precomputed distance matrices | Accuracy ∈ [0, 1] | Works well with DTW/custom distances | Sensitive to scaling unless metric=`precomputed` |
| `randf` | `RandomForestClassifier` | `n_estimators=100`, `max_depth=None`, `random_state=42` | Standard feature matrices | Accuracy ∈ [0, 1] | Robust baseline for tabular embeddings | Fully grown trees; ensemble stabilizes variance |
| `rotf` | `RotationForest + DecisionTreeClassifier` | `n_estimators=100`, PCA‑based rotations | Standard feature matrices | Accuracy ∈ [0, 1] | Strong for time‑series and correlated features | Rotation Forest decorrelates via PCA subsets |
| `shapelet_rotationforest` | `ShapeletTransformClassifier + RotationForest` | `n_shapelet_samples=100`, `max_shapelets=10`, `batch_size=20`, `n_estimators=3` | Raw time‑series arrays | Accuracy ∈ [0, 1] | Captures discriminative subsequences | Best for raw time‑series; slower due to shapelet extraction |
:::


::: {.callout collapse="true" title= "KNN Distance Metrics for the `knn` classifier - Summary Table"}
| Metric name | Meaning / Distance Type | Equation | Input Requirements | Strengths | Notes |
|-------------|--------------------------|----------|--------------------|-----------|-------|
| `euclidean` | Standard L2 distance | $d(x,y) = \sqrt{\sum_i (x_i - y_i)^2}$ | Raw feature vectors | Stable, widely used, works well with normalized data | Equivalent to Minkowski with $p=2$ |
| `nan_euclidean` | NaN‑aware Euclidean distance | Same as Euclidean, computed only over non‑NaN dimensions | Raw features with possible NaNs | Robust when some features are missing | Ignores NaN dimensions; rescales by observed dims |
| `minkowski` | Generalized Lp distance | $d(x,y) = \left(\sum_i \|x_i - y_i\|^p\right)^{1/p}$ | Raw features; requires `p` | Flexible family including L1, L2, L∞ | `p` passed via `calc_knn_similarity(p=...)` |
| `cosine` | Cosine distance (1 − cosine similarity) | $d(x,y) = 1 - \frac{x \cdot y}{\|x\|\|y\|}$ | Raw feature vectors | Good for directional / high‑dimensional data | Scale‑invariant; undefined for zero vectors |
| `l1` | Manhattan (cityblock) distance | $d(x,y) = \sum_i \|x_i - y_i\|$ | Raw feature vectors | Robust to outliers; sparse‑friendly | Equivalent to Minkowski with $p=1$ |
| `precomputed` | User‑supplied distance matrix | *No equation — distances provided externally* | Requires full distance matrices for train/test | Allows DTW, soft‑DTW, custom metrics | `X_train` and `X_test` must be distance matrices |
:::

::: {.callout collapse="true" title= "SVM Kernel Functions for the `svm` classifier - Summary Table"}
| Kernel name | Meaning / Kernel Type | Equation | Strengths | Notes |
|-------------|------------------------|----------|-----------|-------|
| `linear` | Linear kernel (inner product) | $K(x, y) = x^\top y$ | Fast, stable, works well when classes are linearly separable | Equivalent to a linear classifier in feature space |
| `poly` | Polynomial kernel | $K(x, y) = (\gamma\, x^\top y + r)^d$ | Captures polynomial feature interactions | Degree $d$, scale $\gamma$, and offset $r$ correspond to `degree`, `gamma`, and `coef0` in `sklearn.svm.SVC` |
| `rbf` | Radial Basis Function (Gaussian) | $K(x, y) = \exp(-\gamma \|x - y\|^2)$ | Very powerful; handles nonlinear boundaries | $\gamma$ controls smoothness; default often works well |
| `sigmoid` | Sigmoid (tanh) kernel | $K(x, y) = \tanh(\gamma\, x^\top y + r)$ | Related to neural network activation functions | Not always positive‑definite; may require tuning |
| `precomputed` | User‑supplied kernel matrix | *No equation — kernel values provided externally* | Allows custom kernels, DTW kernels, similarity matrices | Input must be an $n_\text{train} \times n_\text{train}$ kernel matrix |
:::

#### Classifier Variant Naming Convention

The return value is a dictionary mapping **classifier variants** to accuracy scores.

Each key is constructed as:
`<classifier_name>_`

where:

- KNN expands over **distance metrics**
- SVM expands over **kernel types**
- Other classifiers have a **single variant** (no suffix)

::: {.callout collapse="true" title= "Variant Naming Table"}
| Classifier name | Variants produced | Example keys | Notes |
|-----------------|-------------------|--------------|-------|
| `knn` | One variant per KNN metric in `KNN_METRICS` | `knn_euclidean`, `knn_cosine`, `knn_minkowski`, `knn_precomputed` | Metric name is appended directly |
| `svm` | One variant per SVM kernel in `SVM_KERNELS` | `svm_linear`, `svm_rbf`, `svm_poly`, `svm_sigmoid`, `svm_precomputed` | Kernel name is appended directly |
| `randf` | Single variant | `randf` | No suffix needed |
| `rotf` | Single variant | `rotf` | No suffix needed |
| `shapelet_rotationforest` | Single variant | `shapelet_rotationforest` | No suffix needed |
:::

##### Examples

If all classifiers and all metrics/kernels are evaluated, the result dictionary may look like:

```python
{
    "knn_euclidean": 0.83,
    "knn_cosine": 0.79,
    "knn_precomputed": 0.91,
    "svm_linear": 0.88,
    "svm_rbf": 0.92,
    "randf": 0.90,
    "rotf": 0.94,
    "shapelet_rotationforest": 0.96,
}
```


### Retrieve and visualize 

In [ ]:
#| eval: false

show_doc(depict_accuracy, title_level=4)

---

#### depict_accuracy

```python

def depict_accuracy(
    dirname, # Directory containing the accuracy summary files.
    features:NoneType=None, # If None or empty, a single file `acc_summ.xlsx` is loaded.
If a list is provided, each feature name `f` triggers loading
`acc_summ_{f}.xlsx`.
    filename:str='acc_summ.xlsx', # Filename used when `features` is None. Default is `"acc_summ.xlsx"`.
    statistics:str='median', # Statistic to extract from the summary tables. Options: `"median"`, `"mean"`, `"std"`, `"iqr"` (interquartile range).
    dtw_filter:NoneType=None, # Optional filter restricting which DTW variants are included.
    columns_filter:NoneType=None, # Optional filter restricting which classifier columns are included.
    verbose:bool=False, # If True, prints additional information during loading and filtering.
): # The constructed accuracy plot.


```

*Plot classifier accuracy curves from one or multiple accuracy summary files.*

This function loads accuracy summary tables (in the format produced by the  pipeline's `acc_summ*.xlsx` files), reshapes them into a long format, and
produces publication‑quality accuracy plots. When multiple features or DTW variants are present, the function delegates all color, dash, and marker
assignments to the external `style_registry` module to ensure consistent, reproducible styling across figures.